In [ ]:
!nvidia-smi

In [14]:

import os
import rasterio
import geopandas as gpd
import numpy as np
import pandas as pd
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from tqdm import tqdm

# ── PATHS ─────────────────────────────────────────────────────────────────────
BASE          = "/home/jupyter-228w1a1286/dinesh-data/hackthonndata"
new_base="/home/jupyter-228w1a1286/dinesh-data/hackthonndata/trainingdata"
SHP_FOLDER    = "/home/jupyter-228w1a1286/dinesh-data/hackthonndata/shp-file2" 
OUTPUT        =new_base + "/DATASET"
IMG_OUT       = OUTPUT + "/images"
MASK_OUT      = OUTPUT + "/masks"
COLOR_OUT     = OUTPUT + "/color_masks"
TILE_SIZE     = 512
MIN_MASK_RATIO = 0.001   # FIX 3: skip tiles with <0.1% annotation

os.makedirs(IMG_OUT,   exist_ok=True)
os.makedirs(MASK_OUT,  exist_ok=True)
os.makedirs(COLOR_OUT, exist_ok=True)


# ── CLASS MAP ─────────────────────────────────────────────────────────────────
CLASS_MAP = {
    "Built_Up_Area_typ": 1,
    "Road":              2,
    "Road_Centre_Line":  3,
    "Railway":           4,
    "Bridge":            5,
    "Water_Body_Line":   6,
    "Waterbody_Point":   7,
    "Utility":           8,
    "Utility_Poly_":     9,
}

# ── BUFFER SETTINGS (metres — valid for EPSG:3857) ────────────────────────────
BUFFER_MAP = {
    "Road_Centre_Line": 3,
    "Railway":          4,
    "Water_Body_Line":  3,
    "Bridge":           2,
    "Waterbody_Point":  2,
}

# ── COLOR MAP ─────────────────────────────────────────────────────────────────
COLOR_MAP = {
    0: [0,   0,   0  ],
    1: [255, 0,   0  ],
    2: [255, 255, 0  ],
    3: [255, 165, 0  ],
    4: [128, 0,   128],
    5: [165, 42,  42 ],
    6: [0,   0,   255],
    7: [0,   255, 255],
    8: [0,   255, 0  ],
    9: [255, 105, 180],
}

# ── FIND TIFF FILES ───────────────────────────────────────────────────────────
tiff_files = [f for f in os.listdir(BASE) if f.endswith(".tif")]
print("TIFF files found:", tiff_files)

first_tiff = os.path.join(BASE, tiff_files[0])
with rasterio.open(first_tiff) as src:
    target_crs = src.crs
print("Target CRS:", target_crs)

# ── LOAD & MERGE SHAPEFILES ───────────────────────────────────────────────────
print("\nLoading shapefiles...")
layers = []

for file in os.listdir(SHP_FOLDER):
    if not file.endswith(".shp"):
        continue
    name = file.replace(".shp", "")
    if name not in CLASS_MAP:
        continue

    gdf = gpd.read_file(os.path.join(SHP_FOLDER, file))
    gdf = gdf[gdf.geometry.notnull() & gdf.is_valid].reset_index(drop=True)

    if gdf.crs != target_crs:
        print(f"  Reprojecting {name}: {gdf.crs} → {target_crs}")
        gdf = gdf.to_crs(target_crs)

    if name in BUFFER_MAP:
        print(f"  Buffering {name} by {BUFFER_MAP[name]}m")
        gdf["geometry"] = gdf.geometry.buffer(BUFFER_MAP[name])
        gdf = gdf[gdf.geometry.area > 0]

    gdf["class"] = CLASS_MAP[name]
    layers.append(gdf)
    print(f"  [OK] {name:25s} label={CLASS_MAP[name]}  objects={len(gdf)}")

# Merge all into one GeoDataFrame
merged = gpd.GeoDataFrame(
    pd.concat(layers, ignore_index=True),
    crs=target_crs
)
print(f"\nTotal objects loaded: {len(merged)}")

# ── FIX 1: Build spatial index ONCE per merged GDF ───────────────────────────
sindex = merged.sindex

# ── TILING LOOP ───────────────────────────────────────────────────────────────
global_count  = 0
skipped_empty = 0
skipped_sparse = 0
pixel_stats   = np.zeros(10, dtype=np.int64)

print("\nStarting tile generation...")

for tif in tiff_files:
    print(f"\nProcessing: {tif}")
    tiff_path = os.path.join(BASE, tif)
    prefix    = tif.replace(".tif", "")

    with rasterio.open(tiff_path) as src:
        # Reproject merged GDF to match this specific TIFF if needed
        local_gdf = merged.to_crs(src.crs) if merged.crs != src.crs else merged
        local_sindex = local_gdf.sindex   # FIX 1: spatial index per tiff

        width, height = src.width, src.height
        src_transform = src.transform
        src_nodata    = src.nodata

        xs = range(0, width,  TILE_SIZE)
        ys = range(0, height, TILE_SIZE)

        with tqdm(total=len(xs)*len(ys), desc=prefix) as pbar:
            for x in xs:
                for y in ys:
                    pbar.update(1)

                    # Skip incomplete edge tiles
                    if x + TILE_SIZE > width or y + TILE_SIZE > height:
                        skipped_empty += 1
                        continue

                    window        = Window(x, y, TILE_SIZE, TILE_SIZE)
                    win_transform = rasterio.windows.transform(window, src_transform)
                    bounds        = rasterio.windows.bounds(window, src_transform)
                    tile_geom     = box(*bounds)

                    # FIX 1: fast sindex pre-filter, then precise clip
                    hits    = list(local_sindex.intersection(bounds))
                    if not hits:
                        skipped_empty += 1
                        continue
                    clipped = local_gdf.iloc[hits]
                    clipped = clipped[clipped.intersects(tile_geom)]
                    if clipped.empty:
                        skipped_empty += 1
                        continue

                    # Build shapes list
                    shapes = [
                        (geom, int(val))
                        for geom, val in zip(clipped.geometry, clipped["class"])
                        if geom is not None and not geom.is_empty
                    ]
                    if not shapes:
                        skipped_empty += 1
                        continue

                    # Rasterize integer mask
                    mask = rasterize(
                        shapes,
                        out_shape=(TILE_SIZE, TILE_SIZE),
                        transform=win_transform,
                        fill=0,
                        dtype='uint8',
                    )

                    # FIX 3: skip sparse tiles
                    if (mask > 0).mean() < MIN_MASK_RATIO:
                        skipped_sparse += 1
                        continue

                    # Pixel stats
                    unique, counts = np.unique(mask, return_counts=True)
                    for u, c in zip(unique, counts):
                        if u < 10:
                            pixel_stats[u] += c

                    # Read image
                    img = src.read(window=window)
                    if img.shape[0] >= 3:
                        img = img[[2, 1, 0]]   # BGR → RGB

                    # Color mask
                    color_mask = np.zeros((TILE_SIZE, TILE_SIZE, 3), dtype=np.uint8)
                    for k, v in COLOR_MAP.items():
                        color_mask[mask == k] = v

                    tile_name = f"{prefix}_tile_{global_count:05d}.tif"

                    # FIX 2: explicit profiles — never inherit wrong metadata
                    img_profile = {
                        "driver":    "GTiff",
                        "dtype":     img.dtype,
                        "width":     TILE_SIZE,
                        "height":    TILE_SIZE,
                        "count":     img.shape[0],
                        "crs":       src.crs,
                        "transform": win_transform,
                        "compress":  "lzw",
                    }
                    mask_profile = {**img_profile, "count": 1, "dtype": "uint8"}
                    color_profile = {**img_profile, "count": 3, "dtype": "uint8"}

                    # Save image
                    with rasterio.open(os.path.join(IMG_OUT, tile_name), "w", **img_profile) as dst:
                        dst.write(img)

                    # Save numeric mask
                    with rasterio.open(os.path.join(MASK_OUT, tile_name), "w", **mask_profile) as dst:
                        dst.write(mask, 1)

                    # Save color mask
                    with rasterio.open(os.path.join(COLOR_OUT, tile_name), "w", **color_profile) as dst:
                        dst.write(np.transpose(color_mask, (2, 0, 1)))

                    global_count += 1

# ── FINAL REPORT ──────────────────────────────────────────────────────────────
CLASS_NAMES = {
    0:"background", 1:"built_up_area", 2:"road",
    3:"road_centre_line", 4:"railway", 5:"bridge",
    6:"water_body_line", 7:"waterbody_point",
    8:"utility", 9:"utility_poly"
}

print("\n✅ DATASET COMPLETE")
print(f"   Tiles saved    : {global_count}")
print(f"   Skipped empty  : {skipped_empty}")
print(f"   Skipped sparse : {skipped_sparse}")
print("\nPixel distribution:")
for i in range(10):
    pct = 100 * pixel_stats[i] / max(pixel_stats.sum(), 1)
    print(f"  Class {i} ({CLASS_NAMES[i]:20s}): {int(pixel_stats[i]):>12,} px  ({pct:.2f}%)")
print(f"\nImages      → {IMG_OUT}")
print(f"Masks       → {MASK_OUT}")
print(f"Color masks → {COLOR_OUT}")
print("\nUNet: use CrossEntropyLoss with num_classes=10")

TIFF files found: ['28996_NADALA_ORTHO.tif', 'TIMMOWAL_37695_ORI.tif', 'fattu_bhila.tif', 'bagga.tif']
Target CRS: EPSG:32643

Loading shapefiles...
  Reprojecting Utility_Poly_: EPSG:3857 → EPSG:32643
  [OK] Utility_Poly_             label=9  objects=1193
  Reprojecting Waterbody_Point: EPSG:3857 → EPSG:32643
  Buffering Waterbody_Point by 2m
  [OK] Waterbody_Point           label=7  objects=20
  Reprojecting Utility: EPSG:3857 → EPSG:32643
  [OK] Utility                   label=8  objects=1227
  [OK] Built_Up_Area_typ         label=1  objects=27488
  Reprojecting Railway: EPSG:3857 → EPSG:32643
  Buffering Railway by 4m
  [OK] Railway                   label=4  objects=4
  Reprojecting Road: EPSG:3857 → EPSG:32643
  [OK] Road                      label=2  objects=968
  Reprojecting Road_Centre_Line: EPSG:3857 → EPSG:32643
  Buffering Road_Centre_Line by 3m
  [OK] Road_Centre_Line          label=3  objects=1527
  Reprojecting Water_Body_Line: EPSG:3857 → EPSG:32643
  Buffering Water_B

28996_NADALA_ORTHO: 100%|██████████| 2808/2808 [01:20<00:00, 34.76it/s] 



Processing: TIMMOWAL_37695_ORI.tif


TIMMOWAL_37695_ORI: 100%|██████████| 3660/3660 [01:07<00:00, 54.42it/s] 



Processing: fattu_bhila.tif


fattu_bhila: 100%|██████████| 1804/1804 [00:30<00:00, 59.75it/s] 



Processing: bagga.tif


bagga: 100%|██████████| 2385/2385 [00:39<00:00, 60.54it/s] 


✅ DATASET COMPLETE
   Tiles saved    : 4819
   Skipped empty  : 5793
   Skipped sparse : 45

Pixel distribution:
  Class 0 (background          ):  647,435,523 px  (51.25%)
  Class 1 (built_up_area       ):  388,862,220 px  (30.78%)
  Class 2 (road                ):   49,832,874 px  (3.94%)
  Class 3 (road_centre_line    ):  131,981,531 px  (10.45%)
  Class 4 (railway             ):            0 px  (0.00%)
  Class 5 (bridge              ):            0 px  (0.00%)
  Class 6 (water_body_line     ):   44,439,682 px  (3.52%)
  Class 7 (waterbody_point     ):       53,439 px  (0.00%)
  Class 8 (utility             ):           93 px  (0.00%)
  Class 9 (utility_poly        ):      666,574 px  (0.05%)

Images      → /home/jupyter-228w1a1286/dinesh-data/hackthonndata/trainingdata/DATASET/images
Masks       → /home/jupyter-228w1a1286/dinesh-data/hackthonndata/trainingdata/DATASET/masks
Color masks → /home/jupyter-228w1a1286/dinesh-data/hackthonndata/trainingdata/DATASET/color_masks

UNet: us

In [23]:
import shutil
import os

folder_path = "/home/jupyter-228w1a1286/dinesh-data/hackthonndata/newdataset"

if os.path.exists(folder_path):
    shutil.rmtree(folder_path)
    print("✅ DATASET_AUG deleted completely")
else:
    print("❌ Folder not found")

✅ DATASET_AUG deleted completely


In [13]:
import os, glob
SHP_FOLDER = "/home/jupyter-228w1a1286/dinesh-data/hackthonndata/shp-file2"
shps = [f.replace(".shp","") for f in os.listdir(SHP_FOLDER) if f.endswith(".shp")]
print("Shapefiles found:")
for s in shps:
    print(f"  {s}")

Shapefiles found:
  Utility_Poly_
  Waterbody_Point
  Utility
  Built_Up_Area_typ
  Railway
  Road
  Water_Body
  Road_Centre_Line
  Water_Body_Line
  Bridge


In [14]:
import os, glob

BASE = "/home/jupyter-228w1a1286/dinesh-data/hackthonndata"

print("=== Full directory structure ===")
for root, dirs, files in os.walk(BASE):
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    level   = root.replace(BASE, "").count(os.sep)
    if level > 5:
        continue
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:8]:
        print(f"{indent}  {f}")
    if len(files) > 8:
        print(f"{indent}  ... ({len(files)} total)")

# Find satellite image
print("\n=== Satellite / raster images ===")
for ext in ["*.tif", "*.tiff", "*.img", "*.jp2", "*.png", "*.jpg"]:
    found = glob.glob(f"{BASE}/**/{ext}", recursive=True)
    if found:
        print(f"\n{ext} — {len(found)} found:")
        for f in found[:10]:
            print(f"  {f}")

# Find shapefiles
print("\n=== Shapefiles ===")
shps = glob.glob(f"{BASE}/**/*.shp", recursive=True)
print(f"{len(shps)} .shp files found:")
for f in shps:
    print(f"  {f}")

=== Full directory structure ===
hackthonndata/
  28996_NADALA_ORTHO.tif
  gpustats.ipynb
  model-seg.ipynb
  data-augmentation.ipynb
  TIMMOWAL_37695_ORI.tif
  fattu_bhila.tif
  bagga.tif
  trainingdata/
    DATASET/
      images/
        28996_NADALA_ORTHO_tile_01102.tif
        TIMMOWAL_37695_ORI_tile_02550.tif
        bagga_tile_04770.tif
        28996_NADALA_ORTHO_tile_00917.tif
        fattu_bhila_tile_03444.tif
        TIMMOWAL_37695_ORI_tile_02931.tif
        bagga_tile_04815.tif
        TIMMOWAL_37695_ORI_tile_02166.tif
        ... (4819 total)
      color_masks/
        28996_NADALA_ORTHO_tile_01102.tif
        TIMMOWAL_37695_ORI_tile_02550.tif
        bagga_tile_04770.tif
        28996_NADALA_ORTHO_tile_00917.tif
        fattu_bhila_tile_03444.tif
        TIMMOWAL_37695_ORI_tile_02931.tif
        bagga_tile_04815.tif
        TIMMOWAL_37695_ORI_tile_02166.tif
        ... (4819 total)
      masks/
        28996_NADALA_ORTHO_tile_01102.tif
        TIMMOWAL_37695_ORI_tile_02550.

In [15]:
import geopandas as gpd
import rasterio
import os

BASE    = "/home/jupyter-228w1a1286/dinesh-data/hackthonndata"
SHP_DIR = f"{BASE}/shp-file2"
RASTERS = [
    f"{BASE}/28996_NADALA_ORTHO.tif",
    f"{BASE}/TIMMOWAL_37695_ORI.tif",
    f"{BASE}/fattu_bhila.tif",
    f"{BASE}/bagga.tif",
]

SHP_FILES = {
    "background":       None,                              # class 0 — implicit
    "built_up_area":    f"{SHP_DIR}/Built_Up_Area_typ.shp",
    "road":             f"{SHP_DIR}/Road.shp",
    "road_centre_line": f"{SHP_DIR}/Road_Centre_Line.shp",
    "railway":          f"{SHP_DIR}/Railway.shp",
    "bridge":           f"{SHP_DIR}/Bridge.shp",
    "water_body":       f"{SHP_DIR}/Water_Body.shp",
    "water_body_line":  f"{SHP_DIR}/Water_Body_Line.shp",
    "waterbody_point":  f"{SHP_DIR}/Waterbody_Point.shp",
    "utility":          f"{SHP_DIR}/Utility.shp",
    "utility_poly":     f"{SHP_DIR}/Utility_Poly_.shp",
}

# ── CHECK RASTERS ─────────────────────────────────────────────────────────────
print("=== RASTER INFO ===")
for r in RASTERS:
    with rasterio.open(r) as src:
        print(f"\n{os.path.basename(r)}")
        print(f"  Size   : {src.width} x {src.height} px")
        print(f"  Bands  : {src.count}  dtype: {src.dtypes[0]}")
        print(f"  CRS    : {src.crs}")
        print(f"  Bounds : {src.bounds}")

# ── CHECK SHAPEFILES ──────────────────────────────────────────────────────────
print("\n\n=== SHAPEFILE INFO ===")
for name, path in SHP_FILES.items():
    if path is None:
        continue
    try:
        gdf = gpd.read_file(path)
        print(f"\n{name}")
        print(f"  Rows     : {len(gdf)}")
        print(f"  CRS      : {gdf.crs}")
        print(f"  Geom type: {gdf.geom_type.unique().tolist()}")
        print(f"  Columns  : {gdf.columns.tolist()}")
    except Exception as e:
        print(f"\n{name} — ERROR: {e}")

=== RASTER INFO ===

28996_NADALA_ORTHO.tif
  Size   : 26259 x 27390 px
  Bands  : 4  dtype: uint8
  CRS    : EPSG:32643
  Bounds : BoundingBox(left=540378.3252300001, bottom=3489958.00821, right=541176.59883, top=3490790.66421)

TIMMOWAL_37695_ORI.tif
  Size   : 31100 x 30532 px
  Bands  : 4  dtype: uint8
  CRS    : EPSG:32643
  Bounds : BoundingBox(left=508420.6625493903, bottom=3488105.753580704, right=509413.78195939027, top=3489080.734989904)

fattu_bhila.tif
  Size   : 20572 x 22451 px
  Bands  : 4  dtype: uint8
  CRS    : EPSG:3857
  Bounds : BoundingBox(left=8360202.7088, bottom=3726830.7418, right=8360933.9468, top=3727628.7694)

bagga.tif
  Size   : 22846 x 26747 px
  Bands  : 4  dtype: uint8
  CRS    : EPSG:3857
  Bounds : BoundingBox(left=8366153.425, bottom=3726466.3686, right=8366965.4931, top=3727417.0989)


=== SHAPEFILE INFO ===

built_up_area
  Rows     : 27503
  CRS      : EPSG:32643
  Geom type: ['Polygon', 'MultiPolygon']
  Columns  : ['OBJECTID', 'GlobalID', 'Uniq

In [ ]:
!kill -9 1741879

In [ ]:
!ps -fp 1741879

In [1]:
import subprocess

result = subprocess.run(
    ["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"],
    capture_output=True, text=True
)
print("GPU process PIDs:")
for pid in result.stdout.strip().split("\n"):
    print(f"  {pid.strip()}")

GPU process PIDs:
  1796863
  2324705
  2323071


In [2]:
import subprocess, os

# Show current notebook PID
print(f"THIS notebook PID: {os.getpid()}")

# Find all other notebook kernels
result = subprocess.run(["ps", "aux"], capture_output=True, text=True)

print("\n=== OTHER NOTEBOOK KERNELS ===")
for line in result.stdout.split("\n"):
    if "ipykernel" in line and str(os.getpid()) not in line:
        parts = line.split()
        pid  = parts[1]
        cmd  = " ".join(parts[10:])
        print(f"PID: {pid} → {cmd}")

THIS notebook PID: 2357586

=== OTHER NOTEBOOK KERNELS ===
PID: 1776413 → /opt/tljh/user/bin/python -m ipykernel_launcher -f /home/jupyter-228w1a1286/.local/share/jupyter/runtime/kernel-d6a47919-9396-4ece-9280-e2ea5365e8e6.json
PID: 1796863 → /opt/tljh/user/bin/python -m ipykernel_launcher -f /home/jupyter-228w1a1286/.local/share/jupyter/runtime/kernel-583287db-4619-46f4-8efb-12975cd0325b.json
PID: 2323071 → /opt/tljh/user/bin/python -m ipykernel_launcher -f /home/jupyter-228w1a1286/.local/share/jupyter/runtime/kernel-1760d2b6-25ce-4aa6-a5b6-ba3d3e8e9f08.json
PID: 2324705 → /opt/tljh/user/bin/python -m ipykernel_launcher -f /home/jupyter-228w1a1286/.local/share/jupyter/runtime/kernel-1ba6db45-8d1b-4746-abdc-d36417a57264.json
PID: 2328275 → /opt/tljh/user/bin/python -m ipykernel_launcher -f /home/jupyter-228w1a1286/.local/share/jupyter/runtime/kernel-1ba6db45-8d1b-4746-abdc-d36417a57264.json
PID: 2328276 → /opt/tljh/user/bin/python -m ipykernel_launcher -f /home/jupyter-228w1a1286/.loca

In [3]:
import os
import numpy as np
import torch
import rasterio
from rasterio.windows import Window
import segmentation_models_pytorch as smp
from albumentations import Compose, Normalize
from albumentations.pytorch import ToTensorV2
import cv2
from tqdm import tqdm

# ─────────────────────────────────────────────
# CONFIG (MATCH TRAINING)
# ─────────────────────────────────────────────
CFG = {
    "checkpoint": "/home/jupyter-228w1a1286/dinesh-data/hackthonndata/4-class/best_model.pth",
    "encoder": "mit_b3",
    "num_classes": 5,
    "tile_size": 1024,
    "overlap": 256,

    "input_tif": "/home/jupyter-228w1a1286/dinesh-data/hackthonndata/amora.tif",
    "out_dir": "./outputs_4class"
}

os.makedirs(CFG["out_dir"], exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────────────────────────
# CLASS INFO (FROM YOUR CODE)
# ─────────────────────────────────────────────
CLASS_NAMES = [
    "background",
    "built_up_area",
    "road_centre_line",
    "water_body",
    "water_body_line",
]

COLOR_MAP = np.array([
    [0,   0,   0  ],
    [255, 0,   0  ],
    [255, 165, 0  ],
    [0,   0,   255],
    [0,   150, 255],
], dtype=np.uint8)

# ─────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────
model = smp.Segformer(
    encoder_name=CFG["encoder"],
    encoder_weights=None,
    in_channels=3,
    classes=CFG["num_classes"],
).to(DEVICE)

ckpt = torch.load(CFG["checkpoint"], map_location=DEVICE, weights_only=False)
state = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model_state"].items()}
model.load_state_dict(state, strict=False)
model.eval()

print("✅ Model loaded")

# ─────────────────────────────────────────────
# TRANSFORM (MATCH VAL)
# ─────────────────────────────────────────────
transform = Compose([
    Normalize(mean=(0.485, 0.456, 0.406),
              std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ─────────────────────────────────────────────
# TTA (OPTIONAL)
# ─────────────────────────────────────────────
def tta(model, x):
    preds = []
    preds.append(model(x))
    preds.append(torch.flip(model(torch.flip(x, [3])), [3]))
    preds.append(torch.flip(model(torch.flip(x, [2])), [2]))
    return torch.mean(torch.stack(preds), dim=0)

# ─────────────────────────────────────────────
# SLIDING WINDOW
# ─────────────────────────────────────────────
def sliding_window(src):

    H, W = src.height, src.width
    tile = CFG["tile_size"]
    stride = tile - CFG["overlap"]

    output = np.zeros((CFG["num_classes"], H, W), dtype=np.float32)
    count  = np.zeros((H, W), dtype=np.float32)

    with torch.no_grad():
        for y in tqdm(range(0, H, stride), desc="Rows"):
            for x in range(0, W, stride):

                window = Window(x, y,
                                min(tile, W-x),
                                min(tile, H-y))

                # ✅ IMPORTANT: only RGB
                img = src.read([1,2,3], window=window).astype(np.float32)/255.0
                img = np.transpose(img, (1,2,0))

                # padding
                pad_h = tile - img.shape[0]
                pad_w = tile - img.shape[1]
                if pad_h > 0 or pad_w > 0:
                    img = np.pad(img, ((0,pad_h),(0,pad_w),(0,0)))

                tensor = transform(image=img)["image"].unsqueeze(0).to(DEVICE)

                pred = tta(model, tensor)  # or model(tensor)
                pred = torch.softmax(pred, dim=1).cpu().numpy()[0]

                pred = pred[:, :window.height, :window.width]

                output[:, y:y+window.height, x:x+window.width] += pred
                count[y:y+window.height, x:x+window.width] += 1

    output /= count
    return np.argmax(output, axis=0)

# ─────────────────────────────────────────────
# VISUALIZATION
# ─────────────────────────────────────────────
def visualize(original, pred):

    # color mask
    color_mask = COLOR_MAP[pred]

    cv2.imwrite(f"{CFG['out_dir']}/color.png",
                cv2.cvtColor(color_mask, cv2.COLOR_RGB2BGR))

    # overlay
    overlay = original.copy()
    overlay[pred > 0] = color_mask[pred > 0]

    blended = cv2.addWeighted(original, 0.6, overlay, 0.4, 0)

    cv2.imwrite(f"{CFG['out_dir']}/overlay.png",
                cv2.cvtColor(blended, cv2.COLOR_RGB2BGR))

    print("✅ Visualization saved")

# ─────────────────────────────────────────────
# RUN
# ─────────────────────────────────────────────
with rasterio.open(CFG["input_tif"]) as src:

    # ✅ IMPORTANT: only RGB
    original = src.read([1,2,3]).astype(np.uint8)
    original = np.transpose(original, (1,2,0))

    prediction = sliding_window(src)

    # save geotiff
    profile = src.profile
    profile.update(dtype=rasterio.uint8, count=1)

    out_path = f"{CFG['out_dir']}/prediction.tif"
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(prediction.astype(np.uint8), 1)

    visualize(original, prediction)

print(f"✅ Done → {CFG['out_dir']}")

✅ Model loaded


Rows: 100%|██████████| 17/17 [02:07<00:00,  7.47s/it]


✅ Visualization saved
✅ Done → ./outputs_4class
